<a href="https://colab.research.google.com/github/turayusa/homework/blob/main/3_3_Activity_Week_3_Jupyter_Notebook_%E2%80%93_Linear_Regression_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DX799 Data Science Capstone

## Week 3 Jupyter Notebook – Linear Regression 3

### Diabetes Hospitalization Analysis Using Forward Selection, Backward Selection, PCR, and PLSR

**Student:** Abdul Karim Turay  
**Course:** DX799 Data Science Capstone  
**Semester:** Fall 2026  
**Week:** Week 3 – Linear Regression 3  
**Dataset:** Diabetes Hospitalization Dataset  
**Outcome Variable:** `time_in_hospital`

---

### Project Objective

The objective of this Week 3 analysis is to apply and compare additional linear regression methods for predicting hospital length of stay (`time_in_hospital`) among patients with diabetes.

The analysis will evaluate forward selection, backward selection, Principal Component Regression (PCR), and Partial Least Squares Regression (PLSR). Model performance will be assessed using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R-squared (R²).

---

# Table of Contents

### 1. Data Preparation and Baseline Model
- **1A. Data Loading and Preparation**
- **1B. Variable Selection and Data Cleaning**
- **1C. Baseline Linear Regression**

### 2. Forward Selection
- **2A. Forward Selection**
- **2B. Forward Selection Model Evaluation**

### 3. Backward Selection
- **3A. Backward Selection**
- **3B. Backward Selection Model Evaluation**

### 4. Forward vs. Backward Selection Comparison
- **4A. Forward vs. Backward Selection Comparison**

### 5. Principal Component Regression (PCR)
- **5A. Principal Component Regression (PCR)**
- **5B. PCR Component Selection**
- **5C. PCR Model Evaluation**

### 6. Partial Least Squares Regression (PLSR)
- **6A. Partial Least Squares Regression (PLSR)**
- **6B. PLSR Component Selection**
- **6C. PLSR Model Evaluation**

### 7. Final Model Comparison
- **7A. Final Model Comparison**

### 8. Final Model Evaluation and Conclusion
- **8A. Final Model Evaluation**
- **8B. Final Conclusion**

# 1. Data Preparation and Baseline Model

## 1A. Data Loading and Preparation

### Objective

The objective of this section is to load the diabetes hospitalization dataset and examine its structure before performing the Week 3 regression analyses.

Before proceeding with variable selection, data cleaning, forward selection, backward selection, Principal Component Regression (PCR), and Partial Least Squares Regression (PLSR), the dataset will be examined to confirm the number of observations and variables and ensure that it was successfully imported.

In [13]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
import io

# Upload dataset
uploaded = files.upload()

# Get uploaded file name
filename = list(uploaded.keys())[0]

# Load dataset
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Confirm successful data loading
print("File loaded:", filename)
print("Dataset shape:", df.shape)

# Display the first five observations
df.head()

Saving diabetic_data.csv to diabetic_data (1).csv
File loaded: diabetic_data (1).csv
Dataset shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### 1A. Analysis

The diabetes hospitalization dataset was successfully loaded, with 101,766 observations and 50 variables. The initial five observations confirmed that the dataset contains demographic, hospitalization, clinical, and treatment-related data.

The successful import and verification of the dataset serve as the foundation for the Week 3 analyses. The next step is to select the outcome and predictor variables needed for forward selection, backward selection, Principal Component Regression (PCR), and Partial Least Squares Regression (PLSR).

## 1B. Variable Selection and Data Cleaning

### Objective

The objective of this subsection is to select the outcome variable and continuous predictors for the Week 3 regression analyses and check them for missing values.

The outcome variable is `time_in_hospital`. The seven continuous predictors used in earlier regression analyses are retained, allowing the Week 3 approaches to be evaluated using a consistent set of candidate predictors.

In [14]:
# Replace "?" with missing values
df = df.replace("?", np.nan)

# Define outcome and candidate predictors
features = [
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses'
]

# Select Week 3 variables
week3_variables = ['time_in_hospital'] + features

df_week3 = df[week3_variables].copy()

# Display dataset information before cleaning
print("Dataset shape before cleaning:", df_week3.shape)

print("\nMissing values:")
print(df_week3.isnull().sum())

# Remove observations with missing values
df_clean = df_week3.dropna().copy()

print("\nObservations after cleaning:", len(df_clean))
print("Observations removed:", len(df_week3) - len(df_clean))

Dataset shape before cleaning: (101766, 8)

Missing values:
time_in_hospital      0
num_lab_procedures    0
num_procedures        0
num_medications       0
number_outpatient     0
number_emergency      0
number_inpatient      0
number_diagnoses      0
dtype: int64

Observations after cleaning: 101766
Observations removed: 0


### 1B. Analysis

The Week 3 analysis dataset consisted of 101,766 observations and eight variables, including the outcome variable `time_in_hospital` and seven continuous candidate predictors.

There were no missing values found among the specified variables. As a result, all 101,766 observations were retained, and none were removed during data cleaning.

The entire dataset is now available for the Week 3 regression analyses, allowing the same observations and candidate predictors to be used to compare forward selection, backward selection, PCR, and PLSR.

## 1C. Train-Test Split and Baseline Linear Regression

### Objective

The objective of this subsection is to divide the cleaned dataset into training and testing sets and develop a baseline multiple linear regression model.

To improve reproducibility, an 80/20 train-test split is used with a fixed random state. The baseline model contains all seven continuous predictors and serves as a reference for assessing the predictive performance of forward selection, backward selection, PCR, and PLSR.

In [15]:
# Import required functions for model development and evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define predictors and outcome
X = df_clean[features]
y = df_clean['time_in_hospital']

# Split the dataset into 80% training and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Create and fit the baseline linear regression model
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

# Generate predictions
y_pred_baseline = baseline_model.predict(X_test)

# Calculate model performance
baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
baseline_r2 = r2_score(y_test, y_pred_baseline)

# Display results
print("Training observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])

print("\nBaseline Linear Regression Performance")
print("MAE:", round(baseline_mae, 4))
print("RMSE:", round(baseline_rmse, 4))
print("R-squared:", round(baseline_r2, 4))

Training observations: 81412
Testing observations: 20354

Baseline Linear Regression Performance
MAE: 1.9251
RMSE: 2.5226
R-squared: 0.2689


### 1C. Analysis

The dataset was divided into 81,412 training observations and 20,354 testing observations based on an 80/20 train-test split.

The baseline multiple linear regression model resulted in an MAE of 1.9251 days and an RMSE of 2.5226 days. This means that the model's predictions differed from the actual hospital length of stay by about 1.93 days on average.

The R-squared (R²) value was 0.2689, indicating that the seven continuous predictors accounted for approximately 26.9% of the variation in `time_in_hospital`.

These results establish the baseline performance against which the Week 3 forward selection, backward selection, PCR, and PLSR models will be evaluated.

# 2. Forward Selection

## 2A. Forward Feature Selection

### Objective

The objective of this section is to use forward selection to identify a smaller subset of predictors for `time_in_hospital`.

Forward selection starts with no predictors and then adds predictors sequentially based on their contribution to the regression model. The selected subset will then be evaluated using the testing data and compared to the baseline model, which includes all seven predictors.

In [16]:
# Import Sequential Feature Selector
from sklearn.feature_selection import SequentialFeatureSelector

# Create linear regression estimator
forward_estimator = LinearRegression()

# Perform forward selection
forward_selector = SequentialFeatureSelector(
    forward_estimator,
    n_features_to_select="auto",
    direction="forward",
    scoring="r2",
    cv=5
)

# Fit forward selection using training data only
forward_selector.fit(X_train, y_train)

# Identify selected predictors
selected_forward_features = X_train.columns[
    forward_selector.get_support()
].tolist()

print("Total candidate predictors:", len(features))
print("Number of selected predictors:", len(selected_forward_features))

print("\nPredictors selected by Forward Selection:")
for feature in selected_forward_features:
    print("-", feature)

Total candidate predictors: 7
Number of selected predictors: 3

Predictors selected by Forward Selection:
- num_lab_procedures
- num_medications
- number_diagnoses


# 2. Forward Selection

## 2A. Forward Feature Selection

### Objective

The objective of this section is to use forward selection to identify a smaller subset of predictors for `time_in_hospital`.

Forward selection starts with no predictors and then adds predictors sequentially based on their contribution to the regression model. The selected subset will then be evaluated using the testing data and compared to the baseline model, which includes all seven predictors.

## 2B. Forward Selection Model Evaluation

### Objective

The objective of this subsection is to train a multiple linear regression model with the three predictors selected through forward selection and assess its performance on the testing dataset.

The model will be evaluated using MAE, RMSE, and R-squared (R²) and compared to the baseline model, which includes all seven predictors.

In [17]:
# Create training and testing datasets using selected predictors
X_train_forward = X_train[selected_forward_features]
X_test_forward = X_test[selected_forward_features]

# Fit the forward-selection regression model
forward_model = LinearRegression()
forward_model.fit(X_train_forward, y_train)

# Generate predictions
y_pred_forward = forward_model.predict(X_test_forward)

# Calculate performance metrics
forward_mae = mean_absolute_error(y_test, y_pred_forward)
forward_rmse = np.sqrt(mean_squared_error(y_test, y_pred_forward))
forward_r2 = r2_score(y_test, y_pred_forward)

# Compare baseline and forward-selection models
forward_comparison = pd.DataFrame({
    'Model': [
        'Baseline Linear Regression',
        'Forward Selection'
    ],
    'Number of Predictors': [
        len(features),
        len(selected_forward_features)
    ],
    'MAE': [
        baseline_mae,
        forward_mae
    ],
    'RMSE': [
        baseline_rmse,
        forward_rmse
    ],
    'R-squared': [
        baseline_r2,
        forward_r2
    ]
})

print("Baseline vs. Forward Selection Performance")
display(forward_comparison.round(4))

Baseline vs. Forward Selection Performance


,Model,Number of Predictors,MAE,RMSE,R-squared
0,Baseline Linear Regression,7,1.9251,2.5226,0.2689
1,Forward Selection,3,1.9305,2.5300,0.2646


### 2B. Analysis

The forward-selection model had three predictors and produced an MAE of 1.9305 days, an RMSE of 2.5300 days, and an R-squared (R²) value of 0.2646.

In comparison, the baseline model used all seven predictors and had an MAE of 1.9251 days, an RMSE of 2.5226 days, and an R-squared (R²) value of 0.2689.

As a result, reducing the model from seven predictors to three resulted in only a small decrease in predictive performance. The R-squared decreased from 0.2689 to 0.2646, whereas the MAE and RMSE increased slightly.

These results suggest that `num_lab_procedures`, `num_medications`, and `number_diagnoses` capture much of the predictive information contained in the seven-predictor baseline model. Forward selection resulted in a simpler model while maintaining performance similar to the full baseline model.

# 3. Backward Selection

## 3A. Backward Feature Selection

### Objective

The objective of this section is to use backward feature selection to identify a smaller subset of predictors for `time_in_hospital`.

Unlike forward selection, which starts with no predictors and adds predictors, backward selection starts with all candidate predictors and gradually removes predictors that contribute the least to model performance.

In [18]:
# Import Sequential Feature Selector
from sklearn.feature_selection import SequentialFeatureSelector

# Create the linear regression estimator
backward_estimator = LinearRegression()

# Apply backward selection
backward_selector = SequentialFeatureSelector(
    backward_estimator,
    n_features_to_select=3,
    direction='backward',
    scoring='r2',
    cv=5
)

# Fit selector using training data only
backward_selector.fit(X_train, y_train)

# Identify selected predictors
selected_backward_features = X_train.columns[
    backward_selector.get_support()
].tolist()

# Display results
print("Total candidate predictors:", len(features))
print("Number of selected predictors:", len(selected_backward_features))

print("\nPredictors selected by Backward Selection:")
for feature in selected_backward_features:
    print("-", feature)

Total candidate predictors: 7
Number of selected predictors: 3

Predictors selected by Backward Selection:
- num_lab_procedures
- num_medications
- number_diagnoses


### 3A. Analysis

Backward selection reduced the seven original candidate predictors to three: `num_lab_procedures`, `num_medications`, and `number_diagnoses`.

These are the same three predictors selected through forward selection. This agreement indicates that these variables were consistently selected by both feature-selection methods from the seven candidate predictors for predicting hospital length of stay.

The consistency of the two feature-selection procedures provides additional evidence that the other four predictors provide less predictive information when the model is limited to three predictors.

## 3B. Backward Selection Model Performance

### Objective

The objective of this subsection is to fit a multiple linear regression model using only the three predictors selected through backward selection.

The model's predictive performance is evaluated using the same testing dataset and compared to the seven-predictor baseline model.

In [19]:
# Create training and testing datasets using selected predictors
X_train_backward = X_train[selected_backward_features]
X_test_backward = X_test[selected_backward_features]

# Fit linear regression model
backward_model = LinearRegression()
backward_model.fit(X_train_backward, y_train)

# Generate predictions
y_pred_backward = backward_model.predict(X_test_backward)

# Calculate performance metrics
backward_mae = mean_absolute_error(y_test, y_pred_backward)
backward_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_backward)
)
backward_r2 = r2_score(y_test, y_pred_backward)

# Create comparison table
backward_comparison = pd.DataFrame({
    "Model": [
        "Baseline Linear Regression",
        "Backward Selection"
    ],
    "Number of Predictors": [
        len(features),
        len(selected_backward_features)
    ],
    "MAE": [
        baseline_mae,
        backward_mae
    ],
    "RMSE": [
        baseline_rmse,
        backward_rmse
    ],
    "R-squared": [
        baseline_r2,
        backward_r2
    ]
})

print("Baseline vs. Backward Selection Performance")
display(backward_comparison.round(4))

Baseline vs. Backward Selection Performance


,Model,Number of Predictors,MAE,RMSE,R-squared
0,Baseline Linear Regression,7,1.9251,2.5226,0.2689
1,Backward Selection,3,1.9305,2.5300,0.2646


### 3B. Analysis

The backward-selection model, with three predictors, had an MAE of 1.9305 days, RMSE of 2.5300 days, and R² of 0.2646.

Compared with the baseline linear regression model, which used seven predictors and produced an MAE of 1.9251, RMSE of 2.5226, and R² of 0.2689, the backward-selection model showed a small decrease in predictive performance.

However, backward selection lowered the number of predictors from seven to three while keeping fairly similar performance. The chosen predictors were 'num_lab_procedures', 'num_medications', and 'number_diagnoses'.

As a result, forward and backward selection yielded similar subsets of predictors and test-set performance. This shows that these three variables capture the majority of the predictive information offered by the seven original predictors.

# 4. Principal Component Regression (PCR)

## Objective

The objective of this section is to use Principal Component Regression (PCR) to predict `time_in_hospital`.

PCR first standardizes the predictor variables before applying Principal Component Analysis (PCA) to transform the original predictors into a smaller set of uncorrelated principal components. Linear regression is then fitted using these components.

The analysis will assess how much variation in the predictor variables is captured by the principal components and evaluate the predictive performance of PCR using MAE, RMSE, and R².

## 4A. Standardization and Principal Component Analysis

Because PCA is sensitive to differences in measurement scale, the seven predictors are standardized prior to calculating the principal components. To prevent data leakage, the scaler is fitted using only the training data before being applied to the testing data.

PCA is then applied to the standardized training data to assess how much variation in the predictor variables is explained by each principal component.

In [20]:
# Import required libraries
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize predictors
scaler_pcr = StandardScaler()

X_train_scaled = scaler_pcr.fit_transform(X_train)
X_test_scaled = scaler_pcr.transform(X_test)

# Fit PCA using all seven predictors
pca = PCA()
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Calculate explained variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Create results table
pca_variance_table = pd.DataFrame({
    "Principal Component": [
        f"PC{i}" for i in range(1, len(features) + 1)
    ],
    "Explained Variance": explained_variance,
    "Cumulative Variance": cumulative_variance
})

print("PCA Explained Variance")
display(pca_variance_table.round(4))

PCA Explained Variance


,Principal Component,Explained Variance,Cumulative Variance
0,PC1,0.2369,0.2369
1,PC2,0.1953,0.4321
2,PC3,0.1370,0.5691
3,PC4,0.1360,0.7051
4,PC5,0.1166,0.8217
5,PC6,0.1030,0.9247
6,PC7,0.0753,1.0000


### 4A. Analysis

The first principal component (PC1) explains 23.69% of the variation across the seven standardized predictors. The first two components combined explain 43.21%, while the first three explain 56.91%.

The cumulative explained variance increases to 70.51% with four components, 82.17% with five, and 92.47% with six. All seven principal components explain 100% of the predictor variance.

These results suggest that the variation in the predictor data is spread across multiple principal components rather than being concentrated in just one or two. As a result, the predictive performance of PCR should be examined across different numbers of principal components to determine how many components are needed to predict hospital length of stay.

## 4B. PCR Model Performance Across Principal Components

PCR models are fitted using one to seven principal components. The models are evaluated on the testing dataset using MAE, RMSE, and R².

Comparing performance across different numbers of principal components reveals whether dimensionality reduction can retain predictive performance while using fewer transformed features.

In [21]:
# Store PCR results
pcr_results = []

# Evaluate PCR models using 1 through 7 principal components
for n_components in range(1, len(features) + 1):

    # Select the required principal components
    X_train_pcr_subset = X_train_pca[:, :n_components]
    X_test_pcr_subset = X_test_pca[:, :n_components]

    # Fit linear regression model
    pcr_model = LinearRegression()
    pcr_model.fit(X_train_pcr_subset, y_train)

    # Generate predictions
    y_pred_pcr = pcr_model.predict(X_test_pcr_subset)

    # Calculate performance metrics
    pcr_mae = mean_absolute_error(y_test, y_pred_pcr)
    pcr_rmse = np.sqrt(
        mean_squared_error(y_test, y_pred_pcr)
    )
    pcr_r2 = r2_score(y_test, y_pred_pcr)

    # Store results
    pcr_results.append([
        n_components,
        pcr_mae,
        pcr_rmse,
        pcr_r2
    ])

# Create results table
pcr_results_df = pd.DataFrame(
    pcr_results,
    columns=[
        "Number of Components",
        "MAE",
        "RMSE",
        "R-squared"
    ]
)

print("PCR Performance Across Principal Components")
display(pcr_results_df.round(4))

PCR Performance Across Principal Components


,Number of Components,MAE,RMSE,R-squared
0,1,1.9911,2.5975,0.2249
1,2,1.9801,2.5868,0.2312
2,3,1.9617,2.5621,0.2458
3,4,1.9622,2.5623,0.2457
4,5,1.9569,2.5576,0.2485
5,6,1.9547,2.5544,0.2504
6,7,1.9251,2.5226,0.2689


### 4B. Analysis

PCR performance improved overall as more principal components were included in the regression model. Using one principal component resulted in an MAE of 1.9911 days, an RMSE of 2.5975 days, and an R² of 0.249.

Using three components, which captured 56.91% of the predictor variance, resulted in an R² of 0.2458. The model achieved an MAE of 1.9547 days, an RMSE of 2.5544 days, and an R² of 0.2504 with six components, which collectively accounted for 92.47% of the predictor variance.

The seven-component PCR model produced the strongest predictive performance, with an MAE of 1.9251 days, an RMSE of 2.5226 days, and an R² of 0.2689. These values are the same as the baseline linear regression model because retaining all seven principal components preserves all of the information in the original standardized predictors.

Therefore, dimensionality reduction through PCR resulted in some loss of predictive performance when fewer than seven components were retained. In this dataset, PCR did not outperform the baseline seven-predictor linear regression model.

# 5. Partial Least Squares Regression (PLSR)

## Objective

The objective of this section is to use Partial Least Squares Regression (PLSR) to predict `time_in_hospital`.

Unlike PCR, which constructs principal components based solely on variation among the predictor variables, PLSR generates latent components while taking into account the relationship between the predictors and the outcome.

MAE, RMSE, and R² will be used to evaluate the predictive performance of PLSR models with varying numbers of components in predicting `time_in_hospital`.

## 5A. PLSR Performance Across Components

PLSR models are fitted using one to seven components. Each model is trained using the standardized training predictors and evaluated on the same testing dataset that was used throughout the analysis.

In [22]:
# Import Partial Least Squares Regression
from sklearn.cross_decomposition import PLSRegression

# Store PLSR results
plsr_results = []

# Evaluate PLSR models using 1 through 7 components
for n_components in range(1, len(features) + 1):

    # Fit PLSR model
    plsr_model = PLSRegression(
        n_components=n_components,
        scale=False
    )

    plsr_model.fit(X_train_scaled, y_train)

    # Generate predictions
    y_pred_plsr = plsr_model.predict(X_test_scaled).ravel()

    # Calculate performance metrics
    plsr_mae = mean_absolute_error(y_test, y_pred_plsr)
    plsr_rmse = np.sqrt(
        mean_squared_error(y_test, y_pred_plsr)
    )
    plsr_r2 = r2_score(y_test, y_pred_plsr)

    # Store results
    plsr_results.append([
        n_components,
        plsr_mae,
        plsr_rmse,
        plsr_r2
    ])

# Create results table
plsr_results_df = pd.DataFrame(
    plsr_results,
    columns=[
        "Number of Components",
        "MAE",
        "RMSE",
        "R-squared"
    ]
)

print("PLSR Performance Across Components")
display(plsr_results_df.round(4))

PLSR Performance Across Components


,Number of Components,MAE,RMSE,R-squared
0,1,1.9446,2.5446,0.2561
1,2,1.9278,2.5247,0.2677
2,3,1.9250,2.5223,0.2691
3,4,1.9251,2.5226,0.2689
4,5,1.9251,2.5226,0.2689
5,6,1.9251,2.5226,0.2689
6,7,1.9251,2.5226,0.2689


### 5A. Analysis

PLSR performance improved as the number of components increased from one to three. The one-component model resulted in an MAE of 1.9446 days, an RMSE of 2.5446 days, and an R² of 0.2561. With two components, R² increased to 0.2677.

The three-component PLSR model had the best test-set performance, with an MAE of 1.9250 days, an RMSE of 2.5223 days, and an R² of 0.2691. This represents a slight improvement over the baseline linear regression model, which had an MAE of 1.9251 days, an RMSE of 2.5226 days, and an R² of 0.2689.

Adding more than three PLSR components did not improve predictive performance. Models with four to seven components produced results nearly identical to the baseline model.

These results indicate that PLSR was able to summarize a substantial portion of the predictive information contained in the seven original predictors using only three latent components. However, the improvement in predictive performance over the baseline model was very small.

# 6. Model Comparison

## Objective

This section compares the predictive performance of the baseline linear regression model, forward and backward selection, PCR, and PLSR.

The comparison determines whether feature selection or component-based regression can reduce model complexity while preserving or improving the prediction of hospital length of stay.

In [23]:
# Select the 3-component PCR results
pcr3 = pcr_results_df[
    pcr_results_df["Number of Components"] == 3
].iloc[0]

# Select the 3-component PLSR results
plsr3 = plsr_results_df[
    plsr_results_df["Number of Components"] == 3
].iloc[0]

# Create final Week 3 comparison table
week3_comparison = pd.DataFrame({
    "Model": [
        "Baseline Linear Regression",
        "Forward Selection",
        "Backward Selection",
        "PCR (3 Components)",
        "PLSR (3 Components)"
    ],
    "Predictors / Components": [
        7,
        3,
        3,
        3,
        3
    ],
    "MAE": [
        baseline_mae,
        forward_mae,
        backward_mae,
        pcr3["MAE"],
        plsr3["MAE"]
    ],
    "RMSE": [
        baseline_rmse,
        forward_rmse,
        backward_rmse,
        pcr3["RMSE"],
        plsr3["RMSE"]
    ],
    "R-squared": [
        baseline_r2,
        forward_r2,
        backward_r2,
        pcr3["R-squared"],
        plsr3["R-squared"]
    ]
})

print("Week 3 Regression Model Comparison")
display(week3_comparison.round(4))

Week 3 Regression Model Comparison


,Model,Predictors / Components,MAE,RMSE,R-squared
0,Baseline Linear Regression,7,1.9251,2.5226,0.2689
1,Forward Selection,3,1.9305,2.5300,0.2646
2,Backward Selection,3,1.9305,2.5300,0.2646
3,PCR (3 Components),3,1.9617,2.5621,0.2458
4,PLSR (3 Components),3,1.9250,2.5223,0.2691


### Analysis

The Week 3 model comparison reveals that the different regression procedures had very similar predictive performance, but reducing the model to three predictors or components had different effects across the methods.

The baseline linear regression model with all seven predictors yielded an MAE of 1.9251 days, an RMSE of 2.5226 days, and an R² of 0.2689.

Forward and backward selection both selected the same three predictors: `num_lab_procedures`, `num_medications`, and `number_diagnoses`. Both models yielded identical results: an MAE of 1.9305 days, an RMSE of 2.5300 days, and an R² of 0.2646. Thus, reducing the original seven predictors to three resulted in only a small decrease in predictive performance.

PCR with three principal components was less effective, with an MAE of 1.9617 days, an RMSE of 2.5621 days, and an R² of 0.2458. The first three principal components captured 56.91% of the predictor variance but did not retain as much information relevant to predicting hospital length of stay.

PLSR with three components resulted in an MAE of 1.9250 days, an RMSE of 2.5223 days, and an R² of 0.2691. These results were marginally better than the baseline model, despite using only three latent components. However, the improvement was very small.

Overall, the results show that model complexity can be reduced while retaining much of the predictive performance. Among the three-component models, PLSR had the strongest predictive performance and most closely preserved the performance of the full seven-predictor baseline model.

# 7. Final Conclusion

Week 3 assessed forward and backward selection, Principal Component Regression (PCR), and Partial Least Squares Regression (PLSR) for predicting `time_in_hospital` in the diabetes hospitalization dataset.

Forward and backward selection resulted in the same reduced model with `num_lab_procedures`, `num_medications`, and `number_diagnoses`. Using these three predictors resulted in only a small decrease in predictive performance compared to the seven-predictor baseline model, suggesting that these variables contain a substantial portion of the predictive information available from the original predictors.

PCR showed that reducing the predictors to a smaller number of principal components resulted in a decrease in predictive performance. Although the first three principal components explained approximately 56.9% of the variation in the predictors, the three-component PCR model produced an R² of 0.2458, compared to 0.2689 for the baseline model.

PLSR outperformed PCR when the number of components was reduced. The three-component PLSR model had an MAE of 1.9250 days, an RMSE of 2.5223 days, and an R² of 0.2691, resulting in predictive performance similar to the full baseline model with a minor numerical improvement.

Overall, the Week 3 results show that dimensionality reduction can simplify the regression model while maintaining similar predictive performance. PLSR performed well in capturing the predictive information from the seven original predictors using only three latent components. However, the R² values remained approximately 0.27, indicating that a substantial portion of the variation in hospital length of stay remains unexplained by the predictors included in this analysis.